# SAE Model Scoping — End-to-End Colab Experiment (Steps 1–7)

This notebook walks through **Steps 1–7** from the README end-to-end on Google Colab,
producing a chemistry-scoped Gemma-2-9B-IT model and evaluating it against two OOD tasks.

**What you will do:**
1. ✅ Fork the repo (prerequisite — done on GitHub)
2. ✅ Verify installation
3. Pick an in-domain dataset (chemistry)
4. Calculate SAE firing rates via `scripts/find_firing_rates.py`
5. Evaluate SAE-hooked model at varying K (inline)
6. Train the LLM with pruned SAE via `scripts/train_with_firing_rates.py`
7. Evaluate finetuned model in-domain and OOD (inline)

> **Tip:** Scripts in `scripts/` are reusable CLI tools. This notebook calls them with
> `!python scripts/<name>.py ...` so you can also run each step standalone from the terminal.

## ⚙️ Requirements

| Requirement | Details |
|---|---|
| **GPU** | A100 40 GB recommended (Colab Pro+). V100 16 GB may work with smaller batch. Gemma-2-9B in bfloat16 needs ~18 GB VRAM. |
| **HuggingFace token** | Accept the [Gemma-2-9B-IT licence](https://huggingface.co/google/gemma-2-9b-it) then add `HF_TOKEN` to Colab secrets. |
| **WandB key** | Optional — add `WANDB_API_KEY` to Colab secrets for training curves. |
| **Disk** | ~20 GB for model weights + checkpoints. |

## ⬇️ Installation

In [ ]:
# Clone the repo (skip if already cloned)
# ⚠️  Replace the URL below with YOUR fork's URL
import os
REPO_URL = "https://github.com/Valpip123EMY/SAEScopingMiniproject.git"
if not os.path.exists("/content/SAEScopingMiniproject"):
    !git clone {REPO_URL}
%cd /content/SAEScopingMiniproject
!pip install -e . -q

In [ ]:
# ── Secrets (HuggingFace + WandB) ────────────────────────────────────────────
import os

try:
    from google.colab import userdata
    HF_TOKEN      = userdata.get("HF_TOKEN") or ""
    WANDB_API_KEY = userdata.get("WANDB_API_KEY") or ""
except Exception:
    HF_TOKEN      = ""   # paste your HF token here if not using Colab secrets
    WANDB_API_KEY = ""   # paste your WandB key here if not using Colab secrets

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
else:
    print("⚠️  No HF_TOKEN found — Gemma download will fail unless you are already logged in.")

WANDB_PROJECT = "sae-scoping-colab"
os.environ["WANDB_PROJECT"] = WANDB_PROJECT
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    os.environ["WANDB_MODE"]    = "online"
    print("WandB: online mode")
else:
    os.environ["WANDB_MODE"] = "disabled"
    print("WandB: disabled (no key)")

## Step 1 — Fork the Repo ✅

This step is done outside Colab: fork
[Valpip123EMY/SAEScopingMiniproject](https://github.com/Valpip123EMY/SAEScopingMiniproject)
on GitHub, then update the clone URL in the Install cell above to point to your fork.

## Step 2 — Verify Installation

In [ ]:
# Quick sanity check — if this cell runs without errors the installation is good
import torch
from sae_lens import SAE
from sae_scoping.trainers.sae_enhanced.prune import get_pruned_sae
from sae_scoping.trainers.sae_enhanced.rank import rank_neurons
from sae_scoping.trainers.sae_enhanced.train import train_sae_enhanced_model
from sae_scoping.utils.hooks.pt_hooks import filter_hook_fn, named_forward_hooks
from sae_scoping.utils.hooks.sae import SAEWrapper

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}  |  device: {DEVICE}")
print("All imports OK ✅")

## Step 3 — Pick a Dataset

**In-domain (ID):** `4gate/StemQAMixture` — `chemistry` config.
We format each example as `Question: …\nAnswer: …` for SFT.

**Out-of-domain (OOD):**
- `codeparrot/apps` — Python coding problems (very different domain from chemistry)
- `HuggingFaceH4/ultrachat_200k` — general-purpose chat (broad coverage)

Change `IN_DOMAIN` below to `"biology"` (or another StemQAMixture split) to scope on a
different subject.

In [ ]:
import json
from pathlib import Path

# ── Experiment configuration ──────────────────────────────────────────────
IN_DOMAIN   = "chemistry"   # in-domain dataset name (change to "biology" etc.)
SAE_LAYER   = 31            # which Gemma-2 layer to hook with the SAE
SAE_ID      = f"layer_{SAE_LAYER}/width_16k/canonical"
SAE_RELEASE = "gemma-scope-9b-pt-res-canonical"
MODEL_NAME  = "google/gemma-2-9b-it"
HOOKPOINT   = f"model.layers.{SAE_LAYER}"
MAX_LENGTH  = 512           # token context window (trade-off: quality vs VRAM)

# ── Training hyper-parameters (Step 6) ────────────────────────────────────
MAX_STEPS  = 500    # increase to 1000+ for a full experiment
BATCH_SIZE = 2
ACCUM      = 4      # effective batch = BATCH_SIZE * ACCUM

# ── Paths ─────────────────────────────────────────────────────────────────
REPO_DIR   = Path("/content/SAEScopingMiniproject")
# find_firing_rates.py saves here by default (scripts/.cache relative to the script)
CACHE_DIR  = REPO_DIR / "scripts" / ".cache"
DIST_PATH  = CACHE_DIR / f"ignore_padding_True/{IN_DOMAIN}/{SAE_ID.replace('/', '--')}/distribution.safetensors"
OUTPUT_DIR = REPO_DIR / "outputs_scoped" / IN_DOMAIN   # where training checkpoints go

print(f"In-domain dataset : {IN_DOMAIN}")
print(f"SAE ID            : {SAE_ID}")
print(f"Distribution path : {DIST_PATH}")
print(f"Output dir        : {OUTPUT_DIR}")

In [ ]:
# Preview a few examples from each dataset (no model needed)
from datasets import load_dataset

print("=== In-domain (chemistry) ===")
_chem = load_dataset("4gate/StemQAMixture", "chemistry", split="train[:3]")
for ex in _chem:
    print(f"Q: {ex['question'][:200]}")
    print()

print("=== OOD: apps (coding) ===")
_apps = load_dataset("codeparrot/apps", split="train[:2]")
for ex in _apps:
    print(ex["question"][:200])
    print()

print("=== OOD: ultrachat (chat) ===")
_uc = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft[:2]")
for ex in _uc:
    msgs = ex["messages"]
    user_msg = next((m["content"] for m in msgs if m["role"] == "user"), "")
    print(user_msg[:200])
    print()

## Step 4 — Calculate SAE Firing Rates

`scripts/find_firing_rates.py` loads the model, hooks the SAE at each configured layer,
runs the in-domain dataset through the model, and saves per-feature firing rate
distributions to `scripts/.cache/`.

Key options:
- `-d` / `--datasets` — comma-separated dataset names (default: all four)
- `-n` / `--max-samples` — cap the number of samples per dataset (useful for a quick test run)
- `-b` / `--batch-size` — inference batch size (reduce if you hit OOM)
- `-o` / `--cache-dir` — override the output directory

⏱ **Runtime:** ~15–30 min on an A100 for 200 chemistry examples with 3 SAE layers.

In [ ]:
# ── Step 4a: Compute SAE firing rates ────────────────────────────────────────
# Runs entirely as a subprocess — no GPU memory used in this notebook process.
# Remove --max-samples for a full run (uses the entire training split).
!python {REPO_DIR}/scripts/find_firing_rates.py \
    --datasets {IN_DOMAIN} \
    --ignore_paddings True \
    --batch-size {BATCH_SIZE} \
    --max-samples 200 \
    --cache-dir {CACHE_DIR}

print()
print("Firing rates saved to:", DIST_PATH)

In [ ]:
# ── Step 4b: Plot SAE firing rate distributions ───────────────────────────────
!python {REPO_DIR}/scripts/plot_firing_rates.py \
    --cache-dir {CACHE_DIR} \
    --ignore-padding True

print()
print("Plots saved to:", CACHE_DIR / "plots")

In [ ]:
# Display the generated plots inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

plot_dir = CACHE_DIR / "plots"
plots = sorted(plot_dir.glob("*.png")) if plot_dir.exists() else []
if not plots:
    print("No plots found — did Step 4b complete successfully?")
else:
    for path in plots:
        img = mpimg.imread(path)
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(path.name)
        plt.tight_layout()
        plt.show()

## Step 5 — Evaluate SAE-Hooked Model at Varying K

We sweep over a range of K values (neurons retained) and measure the average
cross-entropy loss on the chemistry test set.  We want to find the **smallest K**
where loss stays close to the no-SAE baseline — that is the most aggressive pruning
we can apply without hurting in-domain performance.

> This step uses inline Python because the interactive K-sweep is not yet wrapped
> in a standalone script. The model is loaded here and freed at the end of Step 5
> to release GPU memory before the training script runs in Step 6.

In [ ]:
import gc
from contextlib import nullcontext
from functools import partial

import torch
from safetensors.torch import load_file
from sae_lens import SAE
from transformers import AutoTokenizer, Gemma2ForCausalLM, PreTrainedTokenizerBase

from sae_scoping.trainers.sae_enhanced.prune import get_pruned_sae
from sae_scoping.trainers.sae_enhanced.rank import rank_neurons
from sae_scoping.utils.hooks.pt_hooks import filter_hook_fn, named_forward_hooks
from sae_scoping.utils.hooks.sae import SAEWrapper

# ── Load tokenizer + model ────────────────────────────────────────────────
print("Loading tokeniser …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN or None)
assert isinstance(tokenizer, PreTrainedTokenizerBase)

print("Loading model (this may take a few minutes) …")
model = Gemma2ForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="cpu",
    attn_implementation="eager",
    token=HF_TOKEN or None,
)
model = model.to(DEVICE)
model.eval()
print("Model ready.")

In [ ]:
from datasets import load_dataset

# ── Load SAE ──────────────────────────────────────────────────────────────
print(f"Loading SAE: {SAE_ID} …")
sae = SAE.from_pretrained(release=SAE_RELEASE, sae_id=SAE_ID, device=DEVICE)
sae = sae.to(DEVICE)
print(f"SAE: d_in={sae.cfg.d_in}  d_sae={sae.cfg.d_sae}")

# ── Load cached firing-rate distribution ─────────────────────────────────
assert DIST_PATH.exists(), (
    f"Distribution not found at {DIST_PATH}.\n"
    "Did Step 4 (find_firing_rates.py) complete successfully?"
)
data         = load_file(DIST_PATH)
distribution = data["distribution"].cpu()
ranking      = torch.argsort(distribution, descending=True)
print(f"Distribution loaded: {len(distribution)} features")

# ── Load chemistry test split for evaluation ──────────────────────────────
def _fmt(ex): return {"text": f"Question: {ex['question']}\nAnswer: {ex['answer']}"}
chem_test = (
    load_dataset("4gate/StemQAMixture", "chemistry", split="train")
    .map(_fmt, remove_columns=["question", "answer"])
    .train_test_split(test_size=200, seed=42)["test"]
)
print(f"Chemistry test set: {len(chem_test)} examples")

In [ ]:
# ── Helper: average cross-entropy loss ────────────────────────────────────────
@torch.no_grad()
def compute_avg_loss(
    eval_model,
    eval_texts,
    sae_wrapper=None,
    hookpoint=None,
    max_len=MAX_LENGTH,
    batch_size=2,
    n_eval=50,
):
    ctx_mgr = (
        named_forward_hooks(eval_model, {hookpoint: partial(filter_hook_fn, sae_wrapper)})
        if sae_wrapper is not None
        else nullcontext()
    )
    total_loss, n_batches = 0.0, 0
    with ctx_mgr:
        for i in range(0, min(len(eval_texts), n_eval), batch_size):
            batch = tokenizer(
                eval_texts[i : i + batch_size],
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_len,
            ).to(DEVICE)
            out = eval_model(**batch, labels=batch["input_ids"])
            total_loss += out.loss.item()
            n_batches  += 1
    return total_loss / max(n_batches, 1)


# ── Sweep over K values ────────────────────────────────────────────────────
eval_texts_chem = chem_test["text"][:50]
N_EVAL          = 50

baseline_loss = compute_avg_loss(model, eval_texts_chem, n_eval=N_EVAL)
print(f"Baseline (no SAE): {baseline_loss:.4f}")

full_sae  = get_pruned_sae(sae, ranking.to(DEVICE), K_or_p=len(distribution), T=0.0).to(DEVICE)
full_loss = compute_avg_loss(model, eval_texts_chem, SAEWrapper(full_sae), HOOKPOINT, n_eval=N_EVAL)
print(f"Full SAE (K=16k):  {full_loss:.4f}")

K_VALUES = [50, 100, 250, 500, 1000, 2000, 4000, 8000, len(distribution)]
k_results = {}
for K in K_VALUES:
    pruned  = get_pruned_sae(sae, ranking.to(DEVICE), K_or_p=K, T=0.0).to(DEVICE)
    wrapper = SAEWrapper(pruned)
    loss    = compute_avg_loss(model, eval_texts_chem, wrapper, HOOKPOINT, n_eval=N_EVAL)
    k_results[K] = loss
    print(f"  K={K:<6}  loss={loss:.4f}  Δ={loss - baseline_loss:+.4f}")

In [ ]:
import matplotlib.pyplot as plt

# ── Choose K / threshold ──────────────────────────────────────────────────
BUDGET = 1.05   # accept up to 5 % loss increase vs no-SAE baseline
CHOSEN_K = None
for K in sorted(k_results.keys()):
    if k_results[K] <= BUDGET * baseline_loss:
        CHOSEN_K = K
        break

if CHOSEN_K is None:
    CHOSEN_K = max(k_results.keys())
    print(f"⚠️  All K values exceed budget — defaulting to K={CHOSEN_K}.")
else:
    print(f"✅ Chosen K={CHOSEN_K}  (loss={k_results[CHOSEN_K]:.4f}  budget={BUDGET*baseline_loss:.4f})")

CHOSEN_THRESHOLD = distribution[ranking[CHOSEN_K - 1]].item()
print(f"Threshold at K={CHOSEN_K}: {CHOSEN_THRESHOLD:.2e}")

# Plot K sweep results
fig, ax = plt.subplots(figsize=(8, 4))
ks  = sorted(k_results.keys())
ax.plot(ks, [k_results[k] for k in ks], marker="o", label="SAE loss")
ax.axhline(baseline_loss,          color="green",  linestyle="--", label="Baseline (no SAE)")
ax.axhline(BUDGET * baseline_loss, color="orange", linestyle="--", label=f"Budget (×{BUDGET})")
if CHOSEN_K:
    ax.axvline(CHOSEN_K, color="red", linestyle=":", label=f"Chosen K={CHOSEN_K}")
ax.set_xscale("log")
ax.set_xlabel("K (neurons kept)")
ax.set_ylabel("Avg cross-entropy loss")
ax.set_title(f"Step 5 — Loss vs K  ({IN_DOMAIN})")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Free GPU memory before launching the training script ─────────────────────
# The training script (Step 6) loads the model in its own subprocess, so we must
# release GPU memory here first to avoid OOM errors on a single-GPU machine.
del model, sae, full_sae, pruned
gc.collect()
torch.cuda.empty_cache()
print("GPU memory released — ready for Step 6.")

## Step 6 — Train the LLM with Pruned SAE

`scripts/train_with_firing_rates.py` loads the distribution, creates a pruned SAE that
retains only the K most frequently-firing features, then SFT-trains the layers **after**
the SAE hookpoint on in-domain data (all layers before + including the SAE are frozen).

Key options:
- `-p` / `--dist-path` — path to the `distribution.safetensors` from Step 4
- `-h` / `--threshold` — minimum firing rate to keep a neuron (computed in Step 5)
- `-t` / `--train-on-dataset` — in-domain dataset name (e.g. `chemistry`)
- `-s` / `--max-steps` — training steps (500 for a quick test, 1000+ for full)
- `-o` / `--output-dir` — where checkpoints are saved

⏱ **Runtime:** ~30–90 min on an A100 for 500 steps.

In [ ]:
# ── Step 6: SFT-train LLM layers after SAE ───────────────────────────────────
# All arguments can also be run from the terminal — see scripts/README.md.
!python {REPO_DIR}/scripts/train_with_firing_rates.py \
    --dist-path   {DIST_PATH} \
    --threshold   {CHOSEN_THRESHOLD:.6e} \
    --train-on-dataset {IN_DOMAIN} \
    --batch-size  {BATCH_SIZE} \
    --accum       {ACCUM} \
    --max-steps   {MAX_STEPS} \
    --save-every  {MAX_STEPS} \
    --max-length  {MAX_LENGTH} \
    --output-dir  {OUTPUT_DIR} \
    --wandb-project-name {WANDB_PROJECT}

print()
print("Training complete. Checkpoints saved to:", OUTPUT_DIR)

## Step 7 — Evaluate Finetuned Model In-Domain and OOD

We compare three conditions:

| Model | SAE | Expected in-domain | Expected OOD |
|---|---|---|---|
| Original Gemma-2-9B-IT | None | low loss (capable) | low loss (capable) |
| Original + pruned SAE | K kept | ~same (SAE barely changes it) | higher loss (features pruned) |
| **Scoped** (SFT after SAE) | K kept | **lower / equal** (trained on in-domain) | **highest** (OOD neurons pruned + not recovered) |

A successful scoping result shows the scoped model achieving lower or equal in-domain loss
compared to the original, while having noticeably **higher** OOD loss.

In [ ]:
import gc
import torch
from transformers import AutoTokenizer, Gemma2ForCausalLM
from sae_lens import SAE
from safetensors.torch import load_file
from sae_scoping.trainers.sae_enhanced.prune import get_pruned_sae
from sae_scoping.utils.hooks.sae import SAEWrapper

# ── Locate latest checkpoint ──────────────────────────────────────────────
checkpoints = sorted(OUTPUT_DIR.glob("checkpoint-*"))
assert checkpoints, f"No checkpoints found in {OUTPUT_DIR}. Did Step 6 complete?"
CHECKPOINT_DIR = checkpoints[-1]
print(f"Loading scoped model from: {CHECKPOINT_DIR}")

# ── Load tokenizer and original model ────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN or None)

orig_model = Gemma2ForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="cpu",
    attn_implementation="eager",
    token=HF_TOKEN or None,
)
orig_model = orig_model.to(DEVICE)
orig_model.eval()

# ── Load finetuned (scoped) model from checkpoint ─────────────────────────
scoped_model = Gemma2ForCausalLM.from_pretrained(
    str(CHECKPOINT_DIR),
    torch_dtype=torch.bfloat16,
    device_map="cpu",
    attn_implementation="eager",
)
scoped_model = scoped_model.to(DEVICE)
scoped_model.eval()

# ── Recreate pruned SAE from cached distribution ──────────────────────────
dist_data    = load_file(DIST_PATH)
distribution = dist_data["distribution"].cpu()
ranking      = torch.argsort(distribution, descending=True)

sae = SAE.from_pretrained(release=SAE_RELEASE, sae_id=SAE_ID, device=DEVICE)
sae = sae.to(DEVICE)
pruned_sae_eval = get_pruned_sae(sae, ranking.to(DEVICE), K_or_p=CHOSEN_K, T=0.0).to(DEVICE)
sae_wrapper_eval = SAEWrapper(pruned_sae_eval)

print("Models and SAE loaded ✅")

In [ ]:
import json as _json
from datasets import load_dataset

def _fmt_stemqa(ex):
    return {"text": f"Question: {ex['question']}\nAnswer: {ex['answer']}"}

def _fmt_apps(ex):
    try:
        sols = _json.loads(ex.get("solutions", "[]"))
        sol  = sols[0] if sols else ""
    except (ValueError, TypeError):
        sol = ""
    return {"text": f"Question: {ex['question']}\nSolution: {sol}"}

def _fmt_uc(ex):
    return {"text": "\n".join(
        f"{m.get('role','unknown').capitalize()}: {m.get('content','')}"
        for m in ex["messages"]
    )}

def _split(ds, test_size=200):
    return ds.train_test_split(test_size=min(test_size, len(ds) - 1), seed=42)

chem_test  = _split(load_dataset("4gate/StemQAMixture", "chemistry", split="train").map(_fmt_stemqa, remove_columns=["question","answer"]))["test"]
apps_test  = _split(load_dataset("codeparrot/apps", split="train").map(_fmt_apps, remove_columns=load_dataset("codeparrot/apps", split="train[:1]").column_names))["test"]
uc_test    = _split(load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft").map(_fmt_uc, remove_columns=load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft[:1]").column_names))["test"]

EVAL_SETS = {
    "chemistry  (ID)":  chem_test["text"][:50],
    "apps        (OOD)": apps_test["text"][:50],
    "ultrachat   (OOD)": uc_test["text"][:50],
}
print({k: len(v) for k, v in EVAL_SETS.items()})

In [ ]:
import matplotlib.pyplot as plt
from contextlib import nullcontext
from functools import partial
from sae_scoping.utils.hooks.pt_hooks import filter_hook_fn, named_forward_hooks

@torch.no_grad()
def compute_avg_loss(eval_model, eval_texts, sae_wrapper=None, hookpoint=None,
                     max_len=MAX_LENGTH, batch_size=2, n_eval=50):
    ctx = (
        named_forward_hooks(eval_model, {hookpoint: partial(filter_hook_fn, sae_wrapper)})
        if sae_wrapper is not None else nullcontext()
    )
    total, n = 0.0, 0
    with ctx:
        for i in range(0, min(len(eval_texts), n_eval), batch_size):
            batch = tokenizer(
                eval_texts[i: i + batch_size],
                return_tensors="pt", padding=True, truncation=True, max_length=max_len,
            ).to(DEVICE)
            total += eval_model(**batch, labels=batch["input_ids"]).loss.item()
            n += 1
    return total / max(n, 1)


rows = []
print(f"{'Dataset':<25} {'Original':>12} {'Orig+SAE':>12} {'Scoped+SAE':>12}")
print("─" * 65)
for name, texts in EVAL_SETS.items():
    orig_no_sae = compute_avg_loss(orig_model,   texts, None,             None,      n_eval=50)
    orig_sae    = compute_avg_loss(orig_model,   texts, sae_wrapper_eval, HOOKPOINT, n_eval=50)
    scoped_sae  = compute_avg_loss(scoped_model, texts, sae_wrapper_eval, HOOKPOINT, n_eval=50)
    rows.append({"dataset": name, "original": orig_no_sae, "orig_sae": orig_sae, "scoped_sae": scoped_sae})
    print(f"  {name:<23} {orig_no_sae:>12.4f} {orig_sae:>12.4f} {scoped_sae:>12.4f}")

# ── Bar chart ────────────────────────────────────────────────────────────
names  = [r["dataset"]    for r in rows]
orig   = [r["original"]   for r in rows]
o_sae  = [r["orig_sae"]   for r in rows]
sc_sae = [r["scoped_sae"] for r in rows]

x, w = range(len(names)), 0.28
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([i - w for i in x], orig,   w, label="Original (no SAE)",  color="steelblue")
ax.bar([i     for i in x], o_sae,  w, label="Original + SAE",     color="orange",  hatch="//")
ax.bar([i + w for i in x], sc_sae, w, label="Scoped + SAE",       color="green",   hatch="xx")
ax.set_xticks(list(x))
ax.set_xticklabels(names, rotation=12, ha="right")
ax.set_ylabel("Avg cross-entropy loss")
ax.set_title(f"Step 7 — Scoped model evaluation  (K={CHOSEN_K})")
ax.legend()
plt.tight_layout()
plt.show()

## Summary & Next Steps

### What to look for
- **In-domain (chemistry):** scoped model loss should be ≤ original model.
  If not, try increasing `MAX_STEPS` or lowering `LR` in the config cell.
- **OOD (apps / ultrachat):** scoped model loss should be **higher** than the original.
  A larger gap means more successful scoping.

### Re-running individual steps from the terminal
```bash
# Step 4 — compute firing rates
python scripts/find_firing_rates.py -d chemistry -n 200 -b 4

# Step 4 — plot distributions
python scripts/plot_firing_rates.py

# Step 6 — train with pruned SAE (replace THRESHOLD and OUTPUT_DIR)
python scripts/train_with_firing_rates.py \
    -p scripts/.cache/ignore_padding_True/chemistry/layer_31--width_16k--canonical/distribution.safetensors \
    -h <THRESHOLD> -t chemistry -s 500 -o outputs_scoped/chemistry
```

### Extending the experiment
- Try a different `IN_DOMAIN` dataset (biology, physics, …)
- Increase `MAX_STEPS` to 1000+ for a more thorough recovery
- Add a baseline (pruning or unlearning) — see README §8
- Evaluate on additional OOD tasks (e.g. math, virology)
- Try Gemma-3 + GemmaScope2 (README §11)